# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise / company page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Community / Discuss forum',
   'url': 'https://discuss.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Cloudflare/clef
Updated
3 days ago
•
4.21k
•
1.07k
convaiinnovations/laya
Updated
about 18 hours ago
•
3.75k
•
5.11k
abenzerps/Qwen-Image-2.1-Uncensored-GGUF
Updated
6 days ago
•
1.55M
•
2.99k
Lightricks/LTX-2.5
Updated
1 day ago
•
1.63M
•
6.2k
Cloudflare/clef-flash
Updated
3 days ago


In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 17 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nCloudflare/clef\nUpdated\n3 days ago\n•\n4.21k\n•\n1.07k\nconvaiinnovations/laya\nUpdated\nabout 18 hours ago\n•\n3.

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

## Who We Are  
Hugging Face is the global AI community and collaboration platform at the forefront of machine learning innovation. We empower developers, researchers, and enterprises to build the future of AI through open-source models, datasets, and applications. Our mission is to create an open and ethical AI landscape where the machine learning community can learn, share, and collaborate freely.

## Our Platform  
- **Hugging Face Hub**: A central place hosting over 2 million models and 500,000+ datasets. Anyone can explore, experiment, and contribute to open-source machine learning resources.  
- **Spaces**: Interactive machine learning applications built and shared by the community.  
- **Buckets & Storage**: Robust storage solutions for datasets and models, supporting easy deployment and collaboration.  
- **HuggingChat**: Our conversational AI enabling human-like interactive experiences.  
- **Enterprise Solutions**: Tailored support for companies including Hugging Face PRO, enterprise support, inference providers, and custom inference endpoints to integrate and scale AI efficiently.

## Our Community  
We foster a vibrant, fast-growing global community of ML engineers, scientists, and AI enthusiasts. Members collaborate on projects, exchange ideas on our Discord channels, forums, GitHub, and participate in daily AI paper discussions and blog posts. The community drives some of the most widely adopted open-source ML tools and libraries in the industry.

## Our Culture  
At Hugging Face, we believe in openness, ethical AI, and collaboration. We are passionate about supporting the next generation of AI talent through a nurturing and inclusive environment. Our culture thrives on transparency, learning, and building together with the global AI community.  

## Customers & Use Cases  
Enterprises and startups alike use Hugging Face to deploy cutting-edge AI solutions — from natural language processing to computer vision and beyond. We provide scalable infrastructure and expert support to help teams innovate faster and build reliable AI-driven applications.

## Careers & Join Us  
We invite passionate AI researchers, engineers, developers, and community contributors to join us. Be part of a trailblazing team pushing the boundaries of machine learning technology, contributing to the AI ecosystem, and helping shape the future.  
Explore open roles and apply via our Careers page on the website.

## Contact & Learn More  
- Visit our Website: [huggingface.co](https://huggingface.co)  
- Join the Community: Discord, Forum, GitHub  
- Follow us on Twitter and LinkedIn for updates and news  

**Hugging Face** – Building the future of AI, together.  
Your gateway to open-source machine learning innovation and collaboration.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [35]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is **the AI community building the future** — a groundbreaking collaboration platform where the global machine learning community creates, discovers, and shares models, datasets, and applications. With over **2 million models** and **500,000+ datasets**, Hugging Face supports an open ecosystem that fosters innovation and accelerates the adoption of AI technologies.

Their platform is designed to empower researchers, developers, and enterprises alike, making machine learning accessible, collaborative, and scalable.

---

## What We Offer

- **Models Hub:** Explore and contribute to a vast collection of over two million pre-trained machine learning models across varied AI tasks.

- **Datasets:** Access and host more than half a million datasets, spanning numerous domains and applications.

- **Spaces:** Host AI demos and applications with ease using Spaces, enabling rapid prototyping and sharing.

- **HuggingChat:** Engage with interactive AI-powered chat applications.

- **Inference & Storage:** Enterprise-grade inference endpoints and storage solutions designed for reliability and scalability.

- **Enterprise Solutions & PRO:** Tailored support packages and enhanced features for teams and enterprises.

---

## Community & Collaboration

Hugging Face prides itself as **the home of machine learning**, supporting a vibrant community with:

- Discussion forums, Discord channels, and GitHub repositories fostering open collaboration.

- Daily research paper digests and blog posts to keep the community informed and inspired.

- Hardware resources and educational content to help users learn and build confidently.

The collaborative spirit is at the core of Hugging Face’s ethos, enabling developers worldwide to contribute, innovate, and push the boundaries of AI.

---

## Company Culture

- **Open & Inclusive:** Hugging Face nurtures an open environment where every contributor's input is valued, driving progress through diversity and transparency.

- **Innovation-Driven:** Constantly pushing the envelope of AI research and applications by embracing cutting-edge technology and community insights.

- **User-Centric:** Committed to creating accessible tools and resources that empower users of all expertise levels.

- **Community Empowerment:** Focuses on building and sustaining a strong, engaged community as a key driver of growth and innovation.

---

## Who Uses Hugging Face?

- **Researchers & Academics:** Utilize Hugging Face’s extensive repositories and tools to accelerate AI research and publication.

- **Developers & ML Engineers:** Deploy production-ready models and collaborate on solutions in real-time.

- **Enterprises & Startups:** Scale AI initiatives with robust APIs, inference endpoints, and premium support packages.

- **AI Enthusiasts & Students:** Access learning resources and community support to build AI skills and projects.

---

## Career Opportunities

Hugging Face is continuously growing and looking for passionate individuals who want to shape the future of AI:

- Roles in **Machine Learning Engineering**, **Research**, **Product Management**, **Developer Relations**, and more.

- An environment that supports remote and hybrid work, encourages continuous learning, and values diverse talents.

- Join a mission-driven company at the forefront of AI innovation with direct impact on the global ML community.

To explore current openings and apply, visit their [Careers Page](https://huggingface.co/careers).

---

## Get Involved

- Explore AI Models and Applications: https://huggingface.co/models

- Join the Community on Discord and Forums for collaboration and support.

- Access Documentation and Tutorials for all levels at https://huggingface.co/docs

- Follow their Blog for insights, updates, and research developments.

---

## Brand Identity

- **Colors:** Vibrant yellow (#FFD21E), energetic orange (#FF9D00), and modern gray (#6B7280).

- Hugging Face’s friendly and dynamic branding reflects its commitment to approachable and community-driven AI.

---

**Hugging Face** invites you to be part of the growing AI revolution — whether you're building models, sharing datasets, or launching the next breakthrough AI application. Together, we are building the future of machine learning. 

---

*For more information, visit* [huggingface.co](https://huggingface.co)

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>